# Welcome to PROJECT_NAME! 
---

## How does it work?

This welcome notebook provides two main features:

### 1. Browse and Open Notebooks

You will find below an interactive table with all available notebooks, each designed to facilitate a specific aspect of experiment management. Simply click on any "Open the notebook" button to launch it in JupyterLab.

### 2. Check for Updates

Stay up-to-date with the latest notebook versions! Enter your GitHub Personal Access Token and click "Test Connection" to compare your local notebooks with the latest versions from the GitHub repository. You can update any outdated notebooks directly with a single click.

---

## Available notebooks:

Run the cell below to see the interactive notebook list ⬇️⬇️⬇️

In [ ]:
from IPython.display import display, Markdown, HTML
from ipywidgets import widgets, GridspecLayout
from pathlib import Path
import requests
import base64
import yaml

# Define notebooks with their metadata
notebooks = DICT_OF_NOTEBOOKS

# Create grid: 1 header row + number of notebooks
grid = GridspecLayout(1 + len(notebooks), 3, grid_gap="10px")

# Add custom CSS for styling
display(widgets.HTML("""
<style>
.grid-header {
    background-color: #ff6600 !important;
    color: white !important;
    font-weight: 600 !important;
    padding: 12px !important;
    text-align: center !important;
}
.centered-cell {
    text-align: center !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
}
</style>
"""))

# Header row (centered)
grid[0, 0] = widgets.HTML("<div class='grid-header'><b>Notebook Name</b></div>")
grid[0, 1] = widgets.HTML("<div class='grid-header'><b>Description</b></div>")
grid[0, 2] = widgets.HTML("<div class='grid-header'><b>⬇️ Click to Open the Notebook ⬇️</b></div>")

# Data rows
for idx, nb in enumerate(notebooks, start=1):
    # Name column (centered)
    grid[idx, 0] = widgets.HTML(f"<div class='centered-cell'><strong>{nb['name']}</strong></div>")
    
    # Description column (left-aligned for readability)
    grid[idx, 1] = widgets.HTML(nb['description'])
    
    # Open button (centered)
    open_link = widgets.HTML(
        f'<div class="centered-cell"><a href="{nb["path"]}" target="_blank" style="display:inline-block;padding:8px 16px;background-color:#2196F3;color:white;text-decoration:none;border-radius:4px;font-weight:600;">Open the notebook</a></div>'
    )
    
    grid[idx, 2] = open_link

display(grid)

---

## Do you want to check for updates to the notebooks? 

Then, run the code cell below ⬇️⬇️⬇️


In [ ]:
# Load the local information from all the notebooks
local_latest_version_path = Path("..") / "notebook_latest_versions.yaml"
local_latest_versions = yaml.safe_load(local_latest_version_path.read_text())

# Load the online information (GitHub) from all the notebooks
github_owner = GITHUB_OWNER
github_repo_name = GITHUB_REPO_NAME
github_branch = "main"  # Change to "dev" if you want to check against dev branch


version_file_path = "notebooks/notebook_latest_versions.yaml"
version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}?ref={github_branch}"
version_response = requests.get(version_url)

# Debug: show status and raw response if not 200
if version_response.status_code != 200:
	display(Markdown(f"❌ **GitHub API Error** (HTTP {version_response.status_code})"))
	display(Markdown(f"```\n{version_response.text[:500]}\n```"))
else:
	try:
		online_latest_versions = yaml.safe_load(
			base64.b64decode(version_response.json()['content']).decode('utf-8')
		)
	except Exception as e:
		display(Markdown(f"⚠️ Could not parse version file: {e}"))
		online_latest_versions = local_latest_versions

	# Go through all the notebooks and compare versions
	num_rows = sum(len(v) for v in online_latest_versions.values())
	grid = GridspecLayout(1 + num_rows, 5)

	grid[0, 0] = widgets.HTML("<b>Notebook Name</b>")
	grid[0, 1] = widgets.HTML("<b>Notebook Topic</b>")
	grid[0, 2] = widgets.HTML("<b>Local Version</b>")
	grid[0, 3] = widgets.HTML("<b>Status</b>")
	grid[0, 4] = widgets.HTML("<b>Update?</b>")

	needs_update = False

	idx = 1

	for main_folder in online_latest_versions:
		if main_folder in local_latest_versions:
			for subfolder in online_latest_versions[main_folder]:
				grid[idx, 0] = widgets.HTML(f"{subfolder}")
				grid[idx, 1] = widgets.HTML(f"{main_folder}")
				if subfolder in local_latest_versions.get(main_folder, {}):
					online_version = online_latest_versions[main_folder][subfolder]
					local_version = local_latest_versions[main_folder][subfolder]
					
					grid[idx, 2] = widgets.HTML(f"{local_version}")

					if online_version == local_version:
						grid[idx, 3] = widgets.HTML("✅ Up-to-date")
						grid[idx, 4] = widgets.HTML("-")
					else:
						grid[idx, 3] = widgets.HTML(f"❌ Needs Update to {online_version}")
						
						def button_update(main_folder, subfolder, row_idx):
							def show(button):
								button.disabled = True
								button.description = "Updating..."

								# Download the notebook
								notebook_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/notebooks/{main_folder}/{subfolder}/{subfolder}.ipynb?ref={github_branch}"
								notebook_response = requests.get(notebook_url)

								if notebook_response.status_code == 200:
									try:
										# Decode the notebook content
										notebook_content = base64.b64decode(
											notebook_response.json()['content']
										).decode('utf-8')
										
										# Save to local path
										local_notebook_path = Path(".") / main_folder / f"{subfolder}.ipynb"
										local_notebook_path.parent.mkdir(parents=True, exist_ok=True)
										local_notebook_path.write_text(notebook_content, encoding='utf-8')
										
										# Update the local version YAML
										local_latest_versions[main_folder][subfolder] = online_latest_versions[main_folder][subfolder]
										local_latest_version_path.write_text(yaml.dump(local_latest_versions, default_flow_style=False))
										
										display(f"✅ Updated {subfolder} to version {online_latest_versions[main_folder][subfolder]}")
										
										# Update the grid to show up-to-date
										grid[row_idx, 2] = widgets.HTML(online_latest_versions[main_folder][subfolder])
										grid[row_idx, 3] = widgets.HTML("✅ Up-to-date")
										grid[row_idx, 4] = widgets.HTML("-")
										
									except Exception as e:
										display(f"❌ Error saving notebook: {e}")
										button.disabled = False
										button.description = "Update"
								else:
									display(f"❌ Failed to download notebook (HTTP {notebook_response.status_code})")
									display(notebook_response.text[:300])
									button.disabled = False
									button.description = "Update"
								
							return show

						update_button = widgets.Button(description="Update", disable=False) 

						update_button.on_click(button_update(main_folder, subfolder, row_idx=idx))

						grid[idx, 4] = update_button


						needs_update = True
				else:
					grid[idx, 2] = widgets.HTML("Not Found")
					grid[idx, 3] = widgets.HTML("❌ Missing Notebook")
					needs_update = True

				idx += 1
		else:
			for subfolder in online_latest_versions[main_folder]:
				grid[idx, 0] = widgets.HTML(f"{subfolder}")
				grid[idx, 1] = widgets.HTML(f"{main_folder}")
				grid[idx, 2] = widgets.HTML("Not Found")
				grid[idx, 3] = widgets.HTML("❌ Missing Notebook")
				needs_update = True

				idx += 1

	if needs_update:
		display(Markdown("⚠️ Some notebooks need to be updated."))
	else:
		display(Markdown("✅ All notebooks are up-to-date."))

	display(grid)
